# Taller II — Bloque de Sebastián

Entorno local en VS Code, ingesta de sensores MongoDB y MySQL/REST, controles y conversión de altura.

Desde la raíz del repositorio, ejecutar `uv sync --locked` y seleccionar su `.venv` como kernel. Instalar JDK 17 y configurar `JAVA_HOME` en cada equipo. No copiar la carpeta `.venv` entre sistemas.

Las conexiones se consultan en modo lectura. La URI se obtiene de `MONGODB_URI`, del archivo privado `.local/mongodb.json` o mediante entrada oculta. No incluir ese archivo en el ZIP ni en Git.


In [ ]:
import os
from pathlib import Path
import sys

java_local = Path('/usr/lib/jvm/java-17-openjdk')
if not os.environ.get('JAVA_HOME') and java_local.exists():
    os.environ['JAVA_HOME'] = str(java_local)
os.environ.setdefault('SPARK_LOCAL_IP', '127.0.0.1')
os.environ['PYSPARK_PYTHON'] = sys.executable

import pyspark
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master('local[2]')
         .appName('TallerII_Sebastian')
         .config('spark.sql.session.timeZone', 'America/Santiago')
         .config('spark.sql.shuffle.partitions', '2')
         .getOrCreate())
sc = spark.sparkContext
print('Python:', sys.version.split()[0])
print('PySpark:', pyspark.__version__)
print('Java:', sc._jvm.java.lang.System.getProperty('java.version'))
print('Zona horaria:', spark.conf.get('spark.sql.session.timeZone'))
assert spark.range(10).count() == 10
print('Arranque de Spark verificado')


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/10 22:03:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Python: 3.14.6
PySpark: 4.3.0
Java: 17.0.20.1
Zona horaria: America/Santiago


Arranque de Spark verificado


## Transformación de altura

La entrada `alturaft` se convierte a número y se multiplica por 0,3048. Los valores faltantes se conservan como nulos. Los valores no convertibles también producen nulos y deberán contabilizarse al validar la fuente.

In [ ]:
def agregar_altura_metros(df):
    return (df.withColumn('alturaft', F.expr('try_cast(alturaft AS DOUBLE)'))
              .withColumn('altura_metros', F.col('alturaft') * F.lit(0.3048)))

# Casos de prueba aislados; no se utilizan para completar las 100 filas reales.
prueba_altura = spark.createDataFrame(
    [('conocido', '10'), ('faltante', None), ('invalido', 'sin dato')],
    'caso STRING, alturaft STRING'
)
resultado_prueba = agregar_altura_metros(prueba_altura)
valores = {fila.caso: fila.altura_metros for fila in resultado_prueba.collect()}
assert abs(valores['conocido'] - 3.048) < 1e-9
assert valores['faltante'] is None
assert valores['invalido'] is None
resultado_prueba.show()
print('Conversión de altura verificada')



[Stage 3:>                                                          (0 + 2) / 2]



+--------+--------+-------------+
|    caso|alturaft|altura_metros|
+--------+--------+-------------+
|conocido|    10.0|        3.048|
|faltante|    NULL|         NULL|
|invalido|    NULL|         NULL|
+--------+--------+-------------+

Conversión de altura verificada


## Configuración de fuentes

MongoDB usa `telemetria.lecturas`, según el ejemplo del docente. El endpoint PHP es el acceso a MySQL indicado en el taller; no requiere instalar un servidor MySQL local.

Se seleccionan las 100 filas más recientes por fecha y se desempata por ID. MongoDB garantiza unicidad de `_id`. En REST se eliminan duplicados exactos y se rechazan IDs con registros contradictorios.

Las fechas BSON de MongoDB representan instantes UTC y Spark las muestra en America/Santiago. La fecha de MySQL no declara zona horaria: se conserva como fecha local sin zona (`TIMESTAMP_NTZ`) hasta confirmarla con el docente. No se supone correspondencia temporal entre fuentes.


In [ ]:
def texto_o_nulo(valor):
    return None if valor is None else str(valor)

import json
from getpass import getpass
from datetime import datetime, timezone
import requests
from pymongo import MongoClient
from pyspark.sql import types as T

# Compatible con ejecutar desde la raíz, Taller II o notebooks.
carpeta_taller = None
for base in [Path.cwd(), *Path.cwd().parents]:
    if (base / 'notebooks' / 'sebastian_entorno_sensores.ipynb').exists():
        carpeta_taller = base
        break
    candidata = base / 'sumativa3' / 'Taller II'
    if (candidata / 'notebooks').exists():
        carpeta_taller = candidata
        break

mongo_uri = os.environ.get('MONGODB_URI')
archivo_privado = carpeta_taller / '.local' / 'mongodb.json' if carpeta_taller else None
if not mongo_uri and archivo_privado and archivo_privado.exists():
    mongo_uri = json.loads(archivo_privado.read_text())['uri']
if not mongo_uri:
    mongo_uri = getpass('URI MongoDB del notebook del docente (entrada oculta): ')
if not mongo_uri:
    raise ValueError('Falta la URI de MongoDB.')

MYSQL_URL = 'http://mgtt.comunidadingenieria.cl/obtener_datos.php'
fecha_extraccion_utc = datetime.now(timezone.utc).isoformat()
print('Fuentes configuradas. Extracción UTC:', fecha_extraccion_utc)


Fuentes configuradas. Extracción UTC: 2026-11-11T01:03:09.306822+00:00


## Lectura de MongoDB y conversión de altura

Mapeo: `altitudFt` → `alturaft`, `presionAtm` → `presion_atmosferica`, `ppmCO` → `monoxido_carbono`. Se conserva `presionPa` como `presion_pa`. Los nombres sugieren atm, ppm y Pa; confirmar estas unidades con el docente antes de interpretar resultados.

In [ ]:
globals().pop('df_mongo_bpm', None)
try:
    with MongoClient(mongo_uri, serverSelectionTimeoutMS=15000,
                     connectTimeoutMS=15000, socketTimeoutMS=30000,
                     tz_aware=True) as cliente:
        documentos = list(cliente['telemetria']['lecturas'].find(
            {}, {'altitudFt': 1, 'presionAtm': 1, 'presionPa': 1,
                 'ppmCO': 1, 'fecha': 1}
        ).sort([('fecha', -1), ('_id', -1)]).limit(100))
except Exception as error:
    # No exponer la URI ni el mensaje del servidor en las salidas.
    raise RuntimeError('No se pudo leer MongoDB: ' + type(error).__name__ +
                       '. Revisar acceso, red y URI.') from None

if len(documentos) != 100:
    raise ValueError(f'MongoDB devolvió {len(documentos)} registros; se requieren 100.')
campos_mongo = {'_id', 'fecha', 'altitudFt', 'presionAtm', 'ppmCO'}
if not campos_mongo.issubset(set().union(*(d.keys() for d in documentos))):
    raise ValueError('La colección no contiene todos los campos solicitados.')
if any(not isinstance(d.get('fecha'), datetime) for d in documentos):
    raise ValueError('Revisar fechas BSON faltantes o no válidas antes de seleccionar filas.')

filas_mongo = [(
    str(d['_id']), d['fecha'], texto_o_nulo(d.get('altitudFt')),
    texto_o_nulo(d.get('presionAtm')), texto_o_nulo(d.get('presionPa')),
    texto_o_nulo(d.get('ppmCO'))
) for d in documentos]
schema_mongo = T.StructType([
    T.StructField('id', T.StringType(), False),
    T.StructField('fecha', T.TimestampType(), False),
    *[T.StructField(c, T.StringType(), True) for c in
      ['alturaft', 'presion_atmosferica', 'presion_pa', 'monoxido_carbono']]
])
mongo_original = spark.createDataFrame(filas_mongo, schema_mongo)
df_mongo_bpm = mongo_original
for campo in ['alturaft', 'presion_atmosferica', 'presion_pa', 'monoxido_carbono']:
    fallos = df_mongo_bpm.filter(
        F.col(campo).isNotNull() & F.expr(f'try_cast({campo} AS DOUBLE)').isNull()
    ).count()
    print(f'MongoDB {campo}: valores no convertibles = {fallos}')
    df_mongo_bpm = df_mongo_bpm.withColumn(campo, F.expr(f'try_cast({campo} AS DOUBLE)'))
df_mongo_bpm = agregar_altura_metros(df_mongo_bpm).orderBy(
    F.col('fecha').desc(), F.col('id').desc())


MongoDB alturaft: valores no convertibles = 0
MongoDB presion_atmosferica: valores no convertibles = 0


MongoDB presion_pa: valores no convertibles = 0
MongoDB monoxido_carbono: valores no convertibles = 0


## Lectura de MySQL mediante PHP REST

El ejemplo identifica temperatura en °C y humedad en %. Se renombra `temperatura` como `temperatura_interior`, según el enunciado. Se conserva `fecha_original` para no perder el valor recibido.

In [ ]:
try:
    respuesta = requests.get(MYSQL_URL, timeout=30)
    respuesta.raise_for_status()
    datos_mysql = respuesta.json()
except Exception as error:
    raise RuntimeError('No se pudo leer MySQL/REST: ' + type(error).__name__) from None
if not isinstance(datos_mysql, list) or not datos_mysql:
    raise ValueError('Se esperaba una lista JSON no vacía de registros.')
campos_mysql = {'id', 'temperatura', 'humedad', 'fecha'}
if any(not isinstance(d, dict) or not campos_mysql.issubset(d) for d in datos_mysql):
    raise ValueError('El JSON no coincide con el esquema del notebook base.')

filas_mysql = [tuple(texto_o_nulo(d[c]) for c in ['id','temperatura','humedad','fecha'])
               for d in datos_mysql]
mysql_original = spark.createDataFrame(
    filas_mysql, 'id_original STRING, temperatura_interior STRING, humedad STRING, fecha_original STRING')
df_mysql_tipado = mysql_original
for campo, tipo in [('id_original', 'BIGINT'), ('temperatura_interior', 'DOUBLE'),
                    ('humedad', 'DOUBLE'), ('fecha_original', 'TIMESTAMP_NTZ')]:
    conversion = F.expr(f'try_cast({campo} AS {tipo})')
    fallos = df_mysql_tipado.filter(F.col(campo).isNotNull() & conversion.isNull()).count()
    print(f'MySQL {campo}: valores no convertibles = {fallos}')
    destino = {'id_original': 'id', 'fecha_original': 'fecha'}.get(campo, campo)
    df_mysql_tipado = df_mysql_tipado.withColumn(destino, conversion)
if df_mysql_tipado.filter(F.col('id').isNull() | F.col('fecha').isNull()).count():
    raise ValueError('Hay IDs o fechas no válidos; corregir antes de ordenar y seleccionar.')
sin_duplicados = df_mysql_tipado.dropDuplicates()
if sin_duplicados.groupBy('id').count().filter(F.col('count') > 1).count():
    raise ValueError('Hay IDs con registros contradictorios; revisar la fuente.')
print('Filas recibidas:', len(datos_mysql))
print('Duplicados exactos eliminados:', df_mysql_tipado.count() - sin_duplicados.count())
df_mysql_dht = (sin_duplicados.orderBy(F.col('fecha').desc(), F.col('id').desc())
                .limit(100).drop('id_original'))


MySQL id_original: valores no convertibles = 0
MySQL temperatura_interior: valores no convertibles = 0


MySQL humedad: valores no convertibles = 0
MySQL fecha_original: valores no convertibles = 0


Filas recibidas: 1000


Duplicados exactos eliminados: 0


## Controles de aceptación y traspaso

Ambas fuentes deben producir 100 IDs únicos. Se muestran los nulos, esquema, cobertura y una muestra ordenada. Una medición repetida con distinto ID no se elimina por tener el mismo valor.

In [ ]:
def validar_sensor(nombre, df):
    cantidad = df.count()
    assert cantidad == 100, f'{nombre}: se requieren 100 filas, hay {cantidad}'
    assert df.select('id').distinct().count() == cantidad, f'{nombre}: IDs duplicados'
    assert df.filter(F.col('id').isNull()).count() == 0, f'{nombre}: IDs nulos'
    print(nombre, ': 100 filas y 100 IDs únicos')
    df.printSchema()
    df.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns]).show()
    df.select(F.min('fecha').alias('fecha_min'), F.max('fecha').alias('fecha_max')).show(truncate=False)
    df.orderBy(F.col('fecha').desc(), F.col('id').desc()).show(10, truncate=False)

validar_sensor('df_mysql_dht', df_mysql_dht)
validar_sensor('df_mongo_bpm', df_mongo_bpm)
assert df_mongo_bpm.filter(
    ~F.col('altura_metros').eqNullSafe(F.col('alturaft') * F.lit(0.3048))
).count() == 0
print('Ambos sensores validados. DataFrames disponibles para el traspaso a José.')


df_mysql_dht : 100 filas y 100 IDs únicos
root
 |-- temperatura_interior: double (nullable = true)
 |-- humedad: double (nullable = true)
 |-- fecha_original: string (nullable = true)
 |-- id: long (nullable = true)
 |-- fecha: timestamp_ntz (nullable = true)



+--------------------+-------+--------------+---+-----+
|temperatura_interior|humedad|fecha_original| id|fecha|
+--------------------+-------+--------------+---+-----+
|                   0|      0|             0|  0|    0|
+--------------------+-------+--------------+---+-----+

+-------------------+-------------------+
|fecha_min          |fecha_max          |
+-------------------+-------------------+
|2026-11-10 16:31:58|2026-11-10 17:02:57|
+-------------------+-------------------+



+--------------------+-------+-------------------+------+-------------------+
|temperatura_interior|humedad|fecha_original     |id    |fecha              |
+--------------------+-------+-------------------+------+-------------------+
|24.1                |51.0   |2026-11-10 17:02:57|367608|2026-11-10 17:02:57|
|24.1                |51.2   |2026-11-10 17:02:40|367607|2026-11-10 17:02:40|
|24.1                |51.0   |2026-11-10 17:02:23|367606|2026-11-10 17:02:23|
|24.1                |51.2   |2026-11-10 17:02:06|367605|2026-11-10 17:02:06|
|24.1                |51.2   |2026-11-10 17:01:49|367604|2026-11-10 17:01:49|
|24.1                |50.9   |2026-11-10 17:01:31|367603|2026-11-10 17:01:31|
|24.1                |50.7   |2026-11-10 17:01:14|367602|2026-11-10 17:01:14|
|24.1                |50.7   |2026-11-10 17:00:57|367601|2026-11-10 17:00:57|
|24.1                |50.9   |2026-11-10 17:00:40|367600|2026-11-10 17:00:40|
|24.0                |51.2   |2026-11-10 17:00:20|367599|2026-11

df_mongo_bpm : 100 filas y 100 IDs únicos
root
 |-- id: string (nullable = false)
 |-- fecha: timestamp (nullable = false)
 |-- alturaft: double (nullable = true)
 |-- presion_atmosferica: double (nullable = true)
 |-- presion_pa: double (nullable = true)
 |-- monoxido_carbono: double (nullable = true)
 |-- altura_metros: double (nullable = true)

+---+-----+--------+-------------------+----------+----------------+-------------+
| id|fecha|alturaft|presion_atmosferica|presion_pa|monoxido_carbono|altura_metros|
+---+-----+--------+-------------------+----------+----------------+-------------+
|  0|    0|       0|                  0|         0|               0|            0|
+---+-----+--------+-------------------+----------+----------------+-------------+



+-----------------------+-----------------------+
|fecha_min              |fecha_max              |
+-----------------------+-----------------------+
|2026-11-10 21:31:20.893|2026-11-10 22:02:57.841|
+-----------------------+-----------------------+

+------------------------+-----------------------+--------+-------------------+----------+----------------+------------------+
|id                      |fecha                  |alturaft|presion_atmosferica|presion_pa|monoxido_carbono|altura_metros     |
+------------------------+-----------------------+--------+-------------------+----------+----------------+------------------+
|6af3bec1c1ab6b5957d49804|2026-11-10 22:02:57.841|769.34  |0.972485           |98537.0   |3.84            |234.49483200000003|
|6af3beb0c1ab6b5957d49802|2026-11-10 22:02:40.694|768.5   |0.972485           |98537.0   |3.84            |234.2388          |
|6af3be9fc1ab6b5957d49800|2026-11-10 22:02:23.404|769.62  |0.972455           |98534.0   |3.8             |234.580

## Fuentes y pendientes del equipo

Adaptado de los notebooks del docente `-BaseEjemploMongoDB.ipynb` y `-BaseEjemploMysqlPHP_APIRest.ipynb`, Semana 3, MCDI502. Lectura con PyMongo: https://www.mongodb.com/docs/languages/python/pymongo-driver/current/crud/query/specify-documents-to-return/

Antes de integrar, acordar ciudad y esquema con José y Yerko. Confirmar la zona horaria de MySQL y las unidades de MongoDB. Las fechas de estas fuentes no se deben emparejar por igualdad sin comprobar su cobertura. Si se usa índice ordinal, documentar que solo presenta registros juntos.

El jueves: Sebastián revisa integración y consultas SQL. En el video explica objetivo, arranque local, conexiones, conteos y conversión de altura (0:00–2:45).
